In [ ]:
# 
# EXPERIMENT 1 — Freeze ALL layers, train ONLY text embeddings
#


# !pip install TTS torch torchaudio trainer

import torch
import matplotlib.pyplot as plt
import os

from trainer import Trainer, TrainerArgs
from TTS.tts.configs.xtts_config import XttsConfig
from TTS.tts.datasets import load_tts_samples
from TTS.tts.models.xtts import Xtts


In [ ]:
print("=" * 50)
print("EXPERIMENT 1: Train ONLY Text Embeddings")
print("Everything else FROZEN")
print("=" * 50)

XTTS_CHECKPOINT = "/root/.local/share/tts/tts_models--multilingual--multi-dataset--xtts_v2/"
DATASET_PATH    = "xtts_dataset/"
OUTPUT_PATH     = "exp1_output/"
os.makedirs(OUTPUT_PATH, exist_ok=True)



In [ ]:

config = XttsConfig()
config.load_json(os.path.join(XTTS_CHECKPOINT, "config.json"))

config.output_path     = OUTPUT_PATH
config.epochs          = 10          # small — only embeddings training
config.batch_size      = 4
config.eval_batch_size = 2
config.lr              = 1e-4        # higher LR — only new embeddings
config.print_step      = 50
config.save_step       = 500
config.save_checkpoints = True
config.print_eval      = True

In [ ]:


print(f"Epochs     : {config.epochs}")
print(f"LR         : {config.lr}")
print(f"Batch size : {config.batch_size}")

In [ ]:

##  Load dataset 
train_samples, eval_samples = load_tts_samples(
    datasets=[{
        "formatter"       : "ljspeech",
        "dataset_name"    : "urdu_tts",
        "path"            : DATASET_PATH,
        "meta_file_train" : "train/metadata.csv",
        "meta_file_val"   : "val/metadata.csv",
        "language"        : "ur",
        "ignored_speakers": None,
    }],
    eval_split=True,
)
print(f"\nTrain samples : {len(train_samples)}")
print(f"Val samples   : {len(eval_samples)}")

In [ ]:

## Load model 
model = Xtts.init_from_config(config)
model.load_checkpoint(config, checkpoint_dir=XTTS_CHECKPOINT, eval=False)
print("\nPretrained XTTS loaded ")
for param in model.parameters():
    param.requires_grad = False


In [ ]:


# Unfreeze ONLY text embeddings
unfrozen_layers = []
for name, param in model.named_parameters():
    if "text_embedding" in name:
        param.requires_grad = True
        unfrozen_layers.append(name)

# Count params
total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"\nFreezing Strategy:")
print(f"  Total params     : {total_params:,}")
print(f"  Trainable params : {trainable_params:,}")
print(f"  Frozen params    : {total_params - trainable_params:,}")
print(f"  Trainable %      : {100 * trainable_params / total_params:.4f}%")
print(f"\nUnfrozen layers:")
for l in unfrozen_layers:
    print(f"  → {l}")

In [ ]:
print("\n Starting Experiment 1 training...")

trainer = Trainer(
    TrainerArgs(restore_path=None, skip_train_epoch=False),
    config,
    output_path=OUTPUT_PATH,
    model=model,
    train_samples=train_samples,
    eval_samples=eval_samples,
)

trainer.fit()
print("\nExperiment 1 training complete!")

In [ ]:


# Plot loss curve 
import json

log_path = os.path.join(OUTPUT_PATH, "trainer_0_log.json")
if os.path.exists(log_path):
    with open(log_path) as f:
        logs = json.load(f)

    steps  = [l["step"] for l in logs if "loss" in l]
    losses = [l["loss"] for l in logs if "loss" in l]

    plt.figure(figsize=(10, 4))
    plt.plot(steps, losses, label="Train Loss", color="blue")
    plt.xlabel("Step")
    plt.ylabel("Loss")
    plt.title("Experiment 1 — Text Embeddings Only\nTrain Loss Curve")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()
    print("Loss curve saved ")

print("\n Outputs saved to:", OUTPUT_PATH)
print("  Next: Run exp2_unfreeze_gpt2_top4.py")